## 1. Importing + Constants

In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holidays
import warnings
warnings.filterwarnings('ignore')

SEED = 42
ORIGINAL_DATA_PATH = "../original-data"
PROCESSED_DATA_PATH = "../processed-data"

os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)


## 2. Preprocessing Data

In [6]:
# 1. Tải dữ liệu gốc
sales = pd.read_csv(f'{ORIGINAL_DATA_PATH}/sales.csv', parse_dates=['Date'])
submission = pd.read_csv(f'{ORIGINAL_DATA_PATH}/sample_submission.csv', parse_dates=['Date'])
promotions = pd.read_csv(f'{ORIGINAL_DATA_PATH}/promotions.csv', parse_dates=['start_date', 'end_date'])

# 2. Tạo trục thời gian toàn vẹn (từ Train đến hết Test)
all_dates = pd.date_range(start=sales['Date'].min(), end=submission['Date'].max(), freq='D')
df = pd.DataFrame({'Date': all_dates})

# Đánh dấu tập Train/Test
df = df.merge(sales, on='Date', how='left')
df['Split'] = np.where(df['Date'] <= sales['Date'].max(), 'Train', 'Test')

# 3. Kỹ thuật đặc trưng Thời gian (Calendar Features)
def create_calendar_features(df):
    df['year'] = df['Date'].dt.year
    df['month'] = df['Date'].dt.month
    df['day'] = df['Date'].dt.day
    df['day_of_week'] = df['Date'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
    df['quarter'] = df['Date'].dt.quarter
    df['is_month_start'] = df['Date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['Date'].dt.is_month_end.astype(int)
    
    # Mã hóa vòng (Cyclical encoding) giúp mô hình hiểu tính chu kỳ
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week']/7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week']/7)
    
    return df

df = create_calendar_features(df)

# 4. Thêm các ngày lễ Việt Nam
def add_vietnam_holidays(df):
    years = df['Date'].dt.year.unique().tolist()
    vn_holidays = holidays.VN(years=years)
    
    # 1. Chuẩn hóa chuỗi và mở rộng tập từ khóa nhận diện Tết
    tet_keywords = ['tet', 'tết', 'lunar', 'new year', 'nguyên đán', 'giao thừa']
    
    # 2. Ép kiểu toàn bộ keys về pd.Timestamp để đồng bộ với cột df['Date']
    public_holiday_dates = [pd.to_datetime(date) for date, name in vn_holidays.items()]
    
    tet_dates = [
        pd.to_datetime(date) for date, name in vn_holidays.items() 
        if any(keyword in str(name).lower() for keyword in tet_keywords)
    ]
    
    # 3. So sánh thẳng Timestamp vs Timestamp (Vectorized & Type-safe)
    df['is_public_holiday'] = df['Date'].isin(public_holiday_dates).astype(int)
    df['is_tet'] = df['Date'].isin(tet_dates).astype(int)
    
    # 4. Lễ thương mại Dương lịch cố định
    df['is_commercial_event'] = 0
    commercial_dates = {
        (2, 14): 'Valentine',
        (3, 8): 'International_Womens_Day',
        (6, 1): 'Childrens_Day',
        (10, 20): 'Vietnamese_Womens_Day',
        (11, 20): 'Teachers_Day',
        (12, 24): 'Christmas_Eve',
        (12, 25): 'Christmas_Day'
    }
    
    for (m, d), event in commercial_dates.items():
        df.loc[(df['month'] == m) & (df['day'] == d), 'is_commercial_event'] = 1

    return df

df = add_vietnam_holidays(df)

# 5. Xử lý Khuyến mãi (Chỉ tính những biến biết trước trong tương lai)
def count_active_promos(current_date):
    active = promotions[(promotions['start_date'] <= current_date) & (promotions['end_date'] >= current_date)]
    return len(active)

df['active_promos'] = df['Date'].apply(count_active_promos)

# 6. Tạo Target Lags (Cho tập Train)
# Quan trọng: Trong tập Test, các biến này sẽ được điền bằng phương pháp dự báo đệ quy (Recursive)
# Ở bước preprocessing này, chúng ta chỉ tạo cột để mô hình học.
lags = [1, 7, 14, 30, 365]
for lag in lags:
    df[f'rev_lag_{lag}'] = df['Revenue'].shift(lag)
    df[f'cogs_lag_{lag}'] = df['COGS'].shift(lag)

# 7. Lưu dữ liệu
# Loại bỏ các dòng đầu tiên bị NaN do cơ chế Lag
df = df[df['Date'] >= (sales['Date'].min() + pd.Timedelta(days=365))]

df.to_csv(f'{PROCESSED_DATA_PATH}/final_training_data.csv', index=False)

df.head()

,Date,Revenue,COGS,Split,year,month,day,day_of_week,is_weekend,quarter,...,rev_lag_1,cogs_lag_1,rev_lag_7,cogs_lag_7,rev_lag_14,cogs_lag_14,rev_lag_30,cogs_lag_30,rev_lag_365,cogs_lag_365
365,2013-07-04,2521315.44,2408274.98,Train,2013,7,4,3,0,3,...,8038680.63,7869401.17,7748199.74,7462815.86,7078652.68,5515676.80,5397308.75,4297393.33,5123547.94,3982991.19
366,2013-07-05,2494107.65,2432073.97,Train,2013,7,5,4,0,3,...,2521315.44,2408274.98,6285354.21,6043485.10,5526354.45,4396745.86,6860958.96,5505167.85,2751773.45,2150580.23
367,2013-07-06,2452372.43,2331799.51,Train,2013,7,6,5,1,3,...,2494107.65,2432073.97,9546012.78,9167617.84,5117274.96,4111300.17,5692902.11,4632206.37,3054029.42,2517632.84
368,2013-07-07,2958215.01,2950992.63,Train,2013,7,7,6,1,3,...,2452372.43,2331799.51,7372318.59,7149429.97,5844565.63,4855247.11,3843507.28,2986865.05,2667930.94,2108246.62
369,2013-07-08,2571901.18,2450996.77,Train,2013,7,8,0,0,3,...,2958215.01,2950992.63,6359864.53,6248239.94,5243774.50,4800223.91,3646959.35,2897196.14,2360851.90,1808622.79
